# PDF Related Testing

In [1]:
from pypdf import PdfReader
from os import listdir

files = listdir('/home/kumar/Downloads/chipmunk/data/documents/pdf')
files

['aethertech_policy_v1.pdf',
 'aethertech_policy_v2.pdf',
 'nexacore_policy_v1.pdf',
 'vanguard_policy_v1.pdf']

In [ ]:
file = files[0]
with open(f'/home/kumar/Downloads/chipmunk/data/documents/pdf/{file}', 'rb') as f:
    reader = PdfReader(f)
    for idx, page in enumerate(reader.pages):
        print(f"--- Page {idx + 1} ---")
        print(page.extract_text())
        print(page.)


--- Page 1 ---
AetherTech Solutions Pvt. Ltd. Document ID: ATS-TP-2026-v1.0
Employee Commute & Transport Policy
AetherTech Solutions Pvt. Ltd. (Bengaluru Campus)
Corporate Admin & Transport Desk
Version 1.0 — Effective Date: January 1, 2026
Abstract
This document governs daily employee commute services, night-shift safety protocols,
ad-hoc cab bookings, cost caps, weekend pickups, guest travel, and no-show penalties for
AetherTech Solutions Pvt. Ltd. (Bengaluru Campus). All bookings are managed through
the MoveInSync platform. For queries not covered in this policy, contact the AetherTech
Transport Desk at transport-desk@aethertech.example.com.
1 Eligibility Criteria
1.1 Band and Grade Eligibility
Full-time employees in Bands L1 through L4 are eligible for company-provided shared roster
cabs and shuttles. Employees in Band L5 and above are provided a dedicated chauffeur-driven
vehicle under the Executive Mobility Program and are excluded from shared roster bookings.
Interns and tempora

# Testing ChromaDB Collection

In [17]:
import os
import chromadb
from chromadb.utils import embedding_functions
from pypdf import PdfReader

embedding_function = embedding_functions.ONNXMiniLM_L6_V2()

client = chromadb.PersistentClient(path="../data/chroma_db")

# Create or get collection with the explicit embedding function
collection = client.get_or_create_collection(
    name="policy_documents",
    embedding_function=embedding_function
)
collection

Collection(name=policy_documents)

In [18]:
pdf_dir = "/home/kumar/Downloads/chipmunk/data/documents/pdf"
pdf_files = [f for f in os.listdir(pdf_dir) if f.endswith(".pdf")]

documents = []
metadatas = []
ids = []

for file_name in pdf_files:
    file_path = os.path.join(pdf_dir, file_name)
    reader = PdfReader(file_path)
    for idx, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and text.strip():
            doc_id = f"{file_name}_p{idx + 1}"
            documents.append(text)
            metadatas.append({"source": file_name, "page": idx + 1})
            ids.append(doc_id)

# Add documents to Chroma collection (using upsert to avoid duplicates on re-run)
collection.upsert(documents=documents, metadatas=metadatas, ids=ids)
print(f"Indexed {collection.count()} chunks into '{collection.name}'")

Indexed 12 chunks into 'policy_documents'


In [ ]:
# Query example
query = "What is the policy for night-shift safety and escorts?"
results = collection.query(
    query_texts=[query],
    n_results=2,
)

for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"Source: {meta['source']} (Page {meta['page']}), Distance: {distance:.4f}")
    print(doc, "\n" + "-"*50)

Source: aethertech_policy_v2.pdf (Page 3), Distance: 1.1246
AetherTech Solutions Pvt. Ltd. Document ID: ATS-TP-2026-v2.0
6 Guest Travel Policy
6.1 External Clients and Interview Candidates
Company-arranged transport for external clients, vendors, and on-site interview candidates is
permitted only when hosted by an employee in Band L5 or above. Guests are assigned
dedicated vendor sedans and are never pooled into employee shared rosters.
7 No-Show Penalties
7.1 No-Show Strike Policy
Failure to board a scheduled cab within 5 minutes of arrival at the pickup point, or cancelling
within 3 hours of pickup, constitutes a No-Show. Penalties within a rolling 30-day window
are enforced as follows:
• 1st No-Show: Automated warning email to the employee.
• 2nd No-Show: Monetary penalty of INR 500 deducted from monthly payroll and
manager notification.
• 3rd No-Show: Mandatory 14-day suspension of roster transport privileges plus an
additional INR 500 administrative fine.
8 Governance and Document

## Docling method

In [2]:
from docling.document_converter import DocumentConverter
pdf_dir = "/home/kumar/Downloads/chipmunk/data/documents/pdf"
from os import listdir
pdf_files = [f for f in listdir(pdf_dir) if f.endswith(".pdf")]

from time import time
converter = DocumentConverter()

startTime = time()
for file in pdf_files:
    result = converter.convert(f'{pdf_dir}/{file}')

    print(result.document.export_to_markdown())
print(time()-startTime)

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

## Employee Commute &amp; Transport Policy

AetherTech Solutions Pvt. Ltd. (Bengaluru Campus)

Corporate Admin &amp; Transport Desk

Version 1.0 - Effective Date: January 1, 2026

## Abstract

This document governs daily employee commute services, night-shift safety protocols, ad-hoc cab bookings, cost caps, weekend pickups, guest travel, and no-show penalties for AetherTech Solutions Pvt. Ltd. (Bengaluru Campus). All bookings are managed through the MoveInSync platform. For queries not covered in this policy, contact the AetherTech Transport Desk at transport-desk@aethertech.example.com.

## 1 Eligibility Criteria

## 1.1 Band and Grade Eligibility

Full-time employees in Bands L1 through L4 are eligible for company-provided shared roster cabs and shuttles. Employees in Band L5 and above are provided a dedicated chauffeur-driven vehicle under the Executive Mobility Program and are excluded from shared roster bookings. Interns and temporary contractors are ineligible for daily roster t

## PyMuPDF4LLM (Page-wise / Chunk-wise)

In [ ]:
import pymupdf4llm
from os import listdir
from time import time

pdf_dir = "/home/kumar/Downloads/chipmunk/data/documents/pdf"
pdf_files = [f for f in listdir(pdf_dir) if f.endswith(".pdf")]

# Extract page-wise / chunk-wise markdown with metadata
startTime = time()
for file in pdf_files:
    chunks = pymupdf4llm.to_markdown(f"{pdf_dir}/{file}", page_chunks=True)
    print(f"=== {file} ({len(chunks)} pages) ===")
    
print(time()- startTime)


=== Document parser messages ===
Using RapidOCR for OCR processing.
=== aethertech_policy_v1.pdf (3 pages) ===

=== Document parser messages ===
Using RapidOCR for OCR processing.
=== aethertech_policy_v2.pdf (3 pages) ===

=== Document parser messages ===
Using RapidOCR for OCR processing.
=== nexacore_policy_v1.pdf (3 pages) ===

=== Document parser messages ===
Using RapidOCR for OCR processing.
=== vanguard_policy_v1.pdf (3 pages) ===
2.175996780395508


## LlamaMarkdownReader (LlamaIndex Integration)

In [6]:
import pymupdf4llm
from os import listdir
from time import time

pdf_dir = "/home/kumar/Downloads/chipmunk/data/documents/pdf"
pdf_files = [f for f in listdir(pdf_dir) if f.endswith(".pdf")]

# Initialize reader
reader = pymupdf4llm.LlamaMarkdownReader()

start = time()
all_docs = []
for file in pdf_files:
    # load_data returns a list of LlamaIndex Document objects (1 per page/chunk)
    docs = reader.load_data(f"{pdf_dir}/{file}")
    all_docs.extend(docs)
    print(f"{file}: {len(docs)} page documents loaded")

print(f"Total documents: {len(all_docs)} in {time() - start:.3f}s")

# Inspect sample document
sample = all_docs[0]
print(sample.metadata)
print(sample.text[:250])


Successfully imported LlamaIndex
aethertech_policy_v1.pdf: 3 page documents loaded
aethertech_policy_v2.pdf: 3 page documents loaded
nexacore_policy_v1.pdf: 3 page documents loaded
vanguard_policy_v1.pdf: 3 page documents loaded
Total documents: 12 in 2.035s
{'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': 'LaTeX with hyperref', 'producer': 'pdfTeX-1.40.27', 'creationDate': 'D:20261120062955Z', 'modDate': 'D:20261120062955Z', 'trapped': '', 'encryption': None, 'page': 1, 'total_pages': 3, 'file_path': '/home/kumar/Downloads/chipmunk/data/documents/pdf/aethertech_policy_v1.pdf'}
<u>AetherTech</u> <u>Solutions</u> <u>Pvt.</u> <u>Ltd.</u> <u>Document</u> <u>ID:</u> <u>ATS-TP-2026-v1.0</u>

# **Employee Commute & Transport Policy**


AetherTech Solutions Pvt. Ltd. (Bengaluru Campus)


Corporate Admin & Transport Desk


Version 


## Minimal Page-wise Chunking (para as chunk)

In [17]:
import os
import pymupdf4llm

pdf_path = "/home/kumar/Downloads/chipmunk/data/documents/pdf/aethertech_policy_v1.pdf"
pages = pymupdf4llm.to_markdown(pdf_path, page_chunks=True)
print(f'Pages: {len(pages)}')

chunks = []
for p in pages:
    page_num = p["metadata"]["page_number"]
    paras = [x.strip() for x in p["text"].split("\n\n") if x.strip()]
    
    for idx, para in enumerate(paras):
        chunks.append({
            "text": para,
            "metadata": {"source": os.path.basename(pdf_path), "page": page_num, "chunk": idx + 1}
        })

# Display generated chunks
for c in chunks:
    print(f"--- {c['metadata']} ({len(c['text'])} chars) ---")
    print(c["text"][:150], "\n...")



=== Document parser messages ===
Using RapidOCR for OCR processing.
Pages: 3
--- {'source': 'aethertech_policy_v1.pdf', 'page': 1, 'chunk': 1} (30 chars) ---
AetherTech Solutions Pvt. Ltd. 
...
--- {'source': 'aethertech_policy_v1.pdf', 'page': 1, 'chunk': 2} (29 chars) ---
Document ID: ATS-TP-2026-v1.0 
...
--- {'source': 'aethertech_policy_v1.pdf', 'page': 1, 'chunk': 3} (41 chars) ---
# **Employee Commute & Transport Policy** 
...
--- {'source': 'aethertech_policy_v1.pdf', 'page': 1, 'chunk': 4} (49 chars) ---
AetherTech Solutions Pvt. Ltd. (Bengaluru Campus) 
...
--- {'source': 'aethertech_policy_v1.pdf', 'page': 1, 'chunk': 5} (32 chars) ---
Corporate Admin & Transport Desk 
...
--- {'source': 'aethertech_policy_v1.pdf', 'page': 1, 'chunk': 6} (11 chars) ---
Version 1.0 
...
--- {'source': 'aethertech_policy_v1.pdf', 'page': 1, 'chunk': 7} (31 chars) ---
Effective Date: January 1, 2026 
...
--- {'source': 'aethertech_policy_v1.pdf', 'page': 1, 'chunk': 8} (17 chars) ---
#### **Ab